# 01 Experiment Reproduction

연구 목적별 통합 Notebook — 승인된 매핑, 원본 스크립트 및 결과는 변경하지 않는다. 1차 Fine-Tuning까지 포함. 2차 Fine-Tuning 제외.

## 연구 구성
- [`Fine_Tuning_Reproduction.ipynb`](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning/Fine_Tuning_Reproduction.ipynb) — [실험 기록/결과 위치](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning)

## 사용법
원본 코드 셀은 `%%archive_source`로 등록하는 실제 원본 코드이며, 함수명 충돌을 막기 위해 현재 커널에서 임의 실행하지 않는다. 맨 아래의 실행 셀에서 원본 스크립트/Notebook을 고정 커밋의 **별도 checkout**에서 실행한다. 각 실험의 실행 인자·입력·사전 조건은 위 원본 로그/코드로 확인한다. 기존 로그·결과와 새 출력이 섞이지 않도록 독립 복사본을 사용한다. Original Test/교수님 Test를 포함한 파일은 실행 환경에 두지 않는다. 과거 스크립트가 해당 자료를 요구할 경우 실행하지 않는다. 자동 재실행 및 성능 동등성은 미검증.

원본 실행을 위한 로그 디렉터리를 별도로 지정한다.

In [ ]:
# 공통 실행 제어 — Run All은 원본 코드 등록만 하며 학습하지 않는다.
from pathlib import Path
import hashlib, json, subprocess, sys, datetime
from IPython import get_ipython
SOURCE_CELLS = {}
def archive_source(line, cell):
    SOURCE_CELLS[line.strip()] = cell
get_ipython().register_magic_function(archive_source, magic_kind="cell", magic_name="archive_source")
# 실행 시에만 설정: 기존 작업 트리가 아닌 임시 checkout (git worktree add --detach ... <ref>)
WORKTREE = None
LOG_DIR = None
RUN_PLAN = []  # 예: [("원본/스크립트.py", ["--인자", "값"])]
ALLOW_EXECUTION = False

def blob_sha(data):
    return hashlib.sha1(f"blob {len(data)}\0".encode()+data).hexdigest()

def get_verified_source(path):
    if WORKTREE is None:
        raise ValueError("해당 커밋의 별도 checkout 경로 WORKTREE를 지정하세요.")
    root=Path(WORKTREE).expanduser().resolve()
    entry=SOURCE_INDEX[path]
    head=subprocess.check_output(["git","-C",str(root),"rev-parse","HEAD"],text=True).strip()
    if head != entry["ref"]:
        raise ValueError(f"checkout ref 불일치: {head} != {entry['ref']}")
    p=root/path
    if blob_sha(p.read_bytes()) != entry["sha"]:
        raise ValueError(f"원본 Git blob 불일치: {path}")
    if path in SOURCE_CELLS:
        if p.read_text(encoding="utf-8-sig").rstrip("\n") != SOURCE_CELLS[path].rstrip("\n").lstrip("\ufeff"):
            raise ValueError(f"Notebook 보존 코드와 원본이 다릅니다: {path}")
    return root,p

def show_original(path):
    from IPython.display import display, Code
    _,p=get_verified_source(path)
    display(Code(p.read_text(encoding="utf-8"),language="python" if p.suffix==".py" else "json"))

def execute_original(path, args):
    if not ALLOW_EXECUTION:
        raise RuntimeError("명시적 실행 승인이 필요합니다.")
    root,p=get_verified_source(path)
    if LOG_DIR is None:
        raise ValueError("새 외부 LOG_DIR를 지정하세요.")
    logs=Path(LOG_DIR).expanduser().resolve();logs.mkdir(parents=True,exist_ok=True)
    if logs==root or root in logs.parents:
        raise ValueError("LOG_DIR는 checkout 밖에 두세요.")
    stamp=datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
    if p.suffix==".ipynb":
        cmd=[sys.executable,"-m","jupyter","nbconvert","--to","notebook","--execute",str(p),"--output-dir",str(logs),"--output",f"{p.stem}_{stamp}.ipynb",*args]
    else:
        cmd=[sys.executable,str(p),*args]
    log=logs/f"{p.stem}_{stamp}.log"
    with log.open("x",encoding="utf-8") as f:
        f.write(json.dumps({"source":path,**SOURCE_INDEX[path],"command":cmd,"cwd":str(p.parent)},ensure_ascii=False)+"\n");f.flush()
        result=subprocess.run(cmd,cwd=p.parent,stdout=f,stderr=subprocess.STDOUT)
        f.write(f"\nreturncode={result.returncode}\n")
    print(log)
    result.check_returncode()


In [ ]:
SOURCE_INDEX = {'07_fine_tuning/Fine_Tuning_Reproduction.ipynb': {'ref': 'b5810b1e677fdd4588b0c20df480aa83168df938', 'sha': 'b2d1139666ce6e9eec29c79d79f74b29e9521510'}}

## Fine_Tuning_Reproduction

원본: [07_fine_tuning/Fine_Tuning_Reproduction.ipynb](https://github.com/ryu980920/Machine_Learning/blob/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning/Fine_Tuning_Reproduction.ipynb)  
Git blob: `b2d1139666ce6e9eec29c79d79f74b29e9521510`. [기록 및 결과](https://github.com/ryu980920/Machine_Learning/tree/b5810b1e677fdd4588b0c20df480aa83168df938/07_fine_tuning)

원본 Notebook의 Markdown 및 실행 코드 셀을 순서대로 보존한다. 기존 출력·이미지는 복제하지 않는다. 원본 Colab 환경/실행 순서는 원본 링크 참조.

### 원본 셀 0

# SVC & KNN Fine-Tuning — recorded experiment reproduction
This notebook is assembled from the actual executed cells. It has not been independently rerun top-to-bottom as a new notebook.
Use the exact input archives in the private input bundle, at the path defined below. A separate output directory prevents overwriting previous results.
The 1,280 images are development data for this stage, never fit inputs. Official Original Test and Professor Test are not opened.
Candidate fits and inference timing have separate scripts/recorded result files; do not mix pilot and full scores. Do not publish the private raw evaluation data.

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell1
from pathlib import Path
import os,sys,zipfile,json,io,shutil,hashlib,time,gc,inspect
import numpy as np,pandas as pd,sklearn,joblib
from sklearn.metrics import confusion_matrix,precision_recall_fscore_support
from threadpoolctl import threadpool_limits
p=Path('inputs') if Path('inputs/Final_Validation_Private_Evaluation_Kit_20261126.zip').exists() else Path('/mnt/data')  # change if necessary
kit=zipfile.ZipFile(p/'Final_Validation_Private_Evaluation_Kit_20261126.zip')
z=zipfile.ZipFile(p/'Final_10_Model_Validation_Results.zip')
assert not (p/'07_fine_tuning_reproduction').exists(), 'Choose a new output directory'


### 원본 셀 2

## 0. Input restore and baseline hashes

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell3
import shutil,io
from sklearn.metrics import confusion_matrix
root=p/'07_fine_tuning_reproduction'
root.mkdir(exist_ok=True)
for d in ['00_baseline','01_error_analysis','02_SVC','03_KNN','04_final_comparison','05_final_models','06_reports','runtime']: (root/d).mkdir(exist_ok=True)
rt=root/'runtime'
kit.extractall(rt)
with zipfile.ZipFile(p/'final-validation-development-sources.zip') as zz:
 (rt/'development').mkdir(exist_ok=True)
 for f in ['handmade_training_only.npz','official_development_only.npz','source_audit.json']:zz.extract(f,rt/'development')
import hashlib
for m in ['SVC_Combined','KNN_Combined']:
 d=root/'00_baseline'/m;d.mkdir(exist_ok=True)
 with zipfile.ZipFile(p/f'final-fit-{m}.zip') as zz: zz.extractall(d)
 man=json.loads((d/'fit_manifest.json').read_text())
 with open(d/'model.joblib','rb') as f: h=hashlib.file_digest(f,'sha256').hexdigest()
 assert h==man['checkpoint_sha256']
 for n in ['predictions.csv','confusion_matrix.csv','per_class.csv','validation_metrics.json']:
  (d/n).write_bytes(z.read(m+'/'+n))
 print(m,h)
v=np.load(rt/'handmade_validation_1280.npz'); print(v.files,[(k,v[k].shape) for k in v.files])
sdf=pd.read_csv(root/'00_baseline/SVC_Combined/predictions.csv',dtype={'true':str,'pred':str})
kdf=pd.read_csv(root/'00_baseline/KNN_Combined/predictions.csv',dtype={'true':str,'pred':str})
a=sdf.merge(kdf,on=['collector','index','true'],suffixes=('_svc','_knn'),validate='one_to_one')
assert len(a)==1280
a['svc_wrong']=a.true!=a.pred_svc;a['knn_wrong']=a.true!=a.pred_knn
a.to_csv(root/'01_error_analysis/paired_predictions.csv',index=False)
print(a.groupby(['svc_wrong','knn_wrong']).size())
for m in ['svc','knn']:
 print(m,a[a[m+'_wrong']].groupby(['true','pred_'+m]).size().sort_values(ascending=False).head(20))
 print(a.groupby('collector')[m+'_wrong'].sum())

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell4
import time,joblib,sys,os
sys.path.insert(0,str(rt))
import final_benchmark as fb
from threadpoolctl import threadpool_limits
lim=threadpool_limits(limits=2)
print(fb.environment())
t=time.perf_counter()
Xv=fb.hybrid(v['X'].reshape(-1,784))
print('features',Xv.shape,time.perf_counter()-t)
models={}
for m in ['SVC_Combined','KNN_Combined']:
 models[m]=joblib.load(root/'00_baseline'/m/'model.joblib')
print(models['KNN_Combined']._fit_X.shape, models['SVC_Combined']._gamma)

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell5
from sklearn.metrics import precision_recall_fscore_support
labels=fb.LABELS
y=np.asarray(v['y'],int)
for m,pr in [('SVC_Combined','pred_svc'),('KNN_Combined','pred_knn')]:
 t=time.perf_counter(); pp=models[m].predict(Xv); dur=time.perf_counter()-t
 old=np.array([labels.index(x) for x in a[pr]])
 print(m,dur, (pp==y).sum(),(pp!=old).sum())
 assert np.array_equal(pp,old)
 np.save(root/'00_baseline'/m/'reproduced_pred.npy',pp)

### 원본 셀 6

## 1. Error analysis

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell7
# Error diagnostic including feature geometry and actual decision scores/neighbors
import numpy as np,pandas as pd, time
from sklearn.metrics import precision_recall_fscore_support
er=root/'01_error_analysis'
wrong=np.flatnonzero(a.svc_wrong|a.knn_wrong)
raw=v['X']
# Verify alignment
assert np.array_equal(v['collector_id'],a.collector) and np.array_equal(v['sample_index'],a['index'])
mask=raw>=32
geom=[]
for im,b in zip(raw,mask):
 rr,cc=np.where(b); weights=im.astype(float)*b
 geom.append((rr.max()-rr.min()+1,cc.max()-cc.min()+1,int(b.sum()),float((weights*np.arange(28)[None,:]).sum()/weights.sum()),float((weights*np.arange(28)[:,None]).sum()/weights.sum())))
g=pd.DataFrame(geom,columns=['height','width','foreground_pixels','cx','cy'])
g=pd.concat([a,g],axis=1)
g.to_csv(er/'geometry_all.csv',index=False)
print(g.groupby(['svc_wrong','knn_wrong'])[['height','width','foreground_pixels']].agg(['mean','median']).round(2))
dec=models['SVC_Combined'].decision_function(Xv[wrong])
top=np.argsort(dec,axis=1)[:,::-1]
q=a.iloc[wrong].copy()
q['top1_ovr']=np.array(list(labels))[top[:,0]]
q['top2_ovr']=np.array(list(labels))[top[:,1]]
q['ovr_gap']=dec[np.arange(len(wrong)),top[:,0]]-dec[np.arange(len(wrong)),top[:,1]]
q['true_score']=dec[np.arange(len(wrong)),y[wrong]]
q.to_csv(er/'svc_error_decision_scores.csv',index=False)
dist,inds=models['KNN_Combined'].kneighbors(Xv[wrong],n_neighbors=9)
tr=models['KNN_Combined']._fit_X;ty=models['KNN_Combined']._y
# exact duplicates within each 3-neighbor group
dups=[]
neigh=[]
for j,i in enumerate(wrong):
 dup=len({tr[t].tobytes() for t in inds[j,:3]})<3
 dups.append(dup)
 for rank,(d,idx) in enumerate(zip(dist[j],inds[j]),1):
  neigh.append({'collector':int(a.collector.iloc[i]),'index':int(a['index'].iloc[i]),'true':a.true.iloc[i],'knn_pred':a.pred_knn.iloc[i],'rank':rank,'train_row':int(idx),'distance':float(d),'label':labels[ty[idx]],'source':('official_original' if idx<76800 else 'official_aug' if idx<107520 else 'handmade_original' if idx<110080 else 'handmade_aug' if idx<112640 else 'handmade_repeat')})
pd.DataFrame(neigh).to_csv(er/'knn_error_neighbors.csv',index=False)
q['knn_duplicate_top3']=dups
q.to_csv(er/'error_details.csv',index=False)
print('duplicates on KNN errors',q[q.knn_wrong].knn_duplicate_top3.sum(),'/',q.knn_wrong.sum())
print('true neighbor in top3 among KNN errors',np.sum(np.any(ty[inds[q.knn_wrong,:3]]==y[wrong[q.knn_wrong],None],axis=1)))
print('true neighbor in top9 among KNN errors',np.sum(np.any(ty[inds[q.knn_wrong]]==y[wrong[q.knn_wrong],None],axis=1)))
print('SVC top2 true among wrong',sum(q[q.svc_wrong].true==q[q.svc_wrong].top2_ovr))

### 원본 셀 8

## 2. Predeclared protocol and error contact sheet

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell9
# Save detailed, actual diagnostics and fixed protocol before any candidate runs.
from pathlib import Path
import json, hashlib, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,precision_score,f1_score
proto={
 'status':'predeclared_before_candidate_evaluation',
 'seed':42,'threads':2,'validation_role':'development tuning only; no independent new test available',
 'official_dev_n':19200,'handmade_dev_n':1280,'official_test_used':False,'professor_test_used':False,
 'baseline':{'SVC':{'correct':1199,'official_correct':18967,'precision':0.9395594960462721,'f1':0.9364540514111936},'KNN':{'correct':1182,'official_correct':18856,'precision':0.9261688244348776,'f1':0.9232689402444594}},
 'guardrails':{'correct_strictly_higher':True,'macro_precision_at_least_baseline':True,'macro_f1_strictly_higher':True,'official_max_correct_loss':19,'target_pooled_recall_increase':True,'protect_max_class_correct_loss':2,'protected_classes':['0','8','6'],'time':'SVC <= 10% increase preferred, KNN decrease preferred; full matched measurement required before final selection'},
 'target_classes':['B','D','C'],'knn_additional_diagnostic_classes':['5','6'],
 'SVC':{'C':[.5,1,2],'gamma_scale_multiple':[.5,1,2],'handmade_weight':[1,1.5,2,3,4],'target_weight':[1,1.5,2],'target_extra_aug_per_class_cohort':[0,20,40],'combination':'only independent passing factors'},
 'KNN':{'metric_p':[2,1,1.5,3],'k':[1,3,5,7,9],'alpha':[.5,1,2],'repeat':['baseline','remove_all','original_only','augmented_only','lineage_cap'],'PCA':[1108,512,256,128,64],'combination':'only independent passing factors; recheck k and alpha after PCA'},
 'pilot_rule':'distance functions pilot: 8000 stratified training rows, 320 balanced handmade development queries and 320 official development queries; rank and promote only if pilot macro F1 and accuracy improve, not final evidence; any other pilot due execution limit explicitly marked',
 'stop_policy':'no measured result fabricated; record unexecuted/failed stages, retain baseline until full guardrails and timing checked'
}
(root/'EXPERIMENT_PROTOCOL.json').write_text(json.dumps(proto,ensure_ascii=False,indent=2))
# Save images / class compare
fig,axs=plt.subplots(10,12,figsize=(16,14))
for ax in axs.ravel(): ax.axis('off')
for ax,i in zip(axs.ravel(),wrong):
 ax.imshow(raw[i],cmap='gray',vmin=0,vmax=255)
 ax.set_title(f"{a.collector.iloc[i]}:{a['index'].iloc[i]} {a.true.iloc[i]}->{a.pred_svc.iloc[i]}/{a.pred_knn.iloc[i]}",fontsize=7)
fig.suptitle("All 115 unique errors: true -> SVC / KNN",fontsize=16)
fig.tight_layout();fig.savefig(er/'all_115_error_images.png',dpi=150);plt.close(fig)
for m in ['svc','knn']:
 df=a[a[m+'_wrong']].groupby(['true','pred_'+m]).size().reset_index(name='count').sort_values('count',ascending=False)
 df.to_csv(er/f'{m}_confusions.csv',index=False)
 print(m,df.head(5).to_dict('records'))
print('protocol saved',root)

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell10
import time
t=time.perf_counter()
off,wr=fb.read_training()
Xo=fb.hybrid(off['X_val'])
yo=off['y_val']
print(Xo.shape,'official feature prep',time.perf_counter()-t)
# Cache max 9 full-data KNN neighbors for 1280 and first official chunk; exact native sklearn.
knn=models['KNN_Combined']
t=time.perf_counter()
dv,iv=knn.kneighbors(Xv,n_neighbors=9)
print('dev max9',time.perf_counter()-t)
np.savez_compressed(root/'03_KNN'/'neighbors_handmade_p2_k9.npz',distance=dv,index=iv)

### 원본 셀 11

## 3. KNN vote DOE, full development set

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell12
# KNN scoring and 13 unique vote configs, full 1280
def vote(d,ids,k,alpha,train_y=ty):
    ds=d[:,:k]; ls=train_y[ids[:,:k]]
    zero=ds==0
    weights=np.power(np.maximum(ds,1e-30),-alpha)
    weights=np.where(zero.any(axis=1)[:,None],zero,weights)
    scores=np.zeros((len(ds),16))
    np.add.at(scores,(np.arange(len(ds))[:,None],ls),weights)
    return np.argmax(scores,axis=1)
def stats(pred,base,y=y):
    pp,rr,ff,ss=precision_recall_fscore_support(y,pred,labels=np.arange(16),zero_division=0)
    return {'n':len(y),'correct':int((pred==y).sum()),'accuracy':float(np.mean(pred==y)),'macro_precision':float(pp.mean()),'macro_f1':float(ff.mean()),'corrected':int(np.sum((base!=y)&(pred==y))),'regressed':int(np.sum((base==y)&(pred!=y))),**{f'recall_{c}':float(r) for c,r in zip(labels,rr)}}
kbase=np.load(root/'00_baseline/KNN_Combined/reproduced_pred.npy')
sbase=np.load(root/'00_baseline/SVC_Combined/reproduced_pred.npy')
print('baseline max9 vote mismatch',np.sum(vote(dv,iv,3,1)!=kbase))
krows=[]
kpreds={}
for k in [1,3,5,7,9]:
 for alpha in ([1] if k==1 else [.5,1,2]):
    name=f'k{k}_a{alpha}'
    pred=vote(dv,iv,k,alpha)
    kpreds[name]=pred
    krows.append({'condition':name,'k':k,'alpha':alpha,**stats(pred,kbase)})
kr=pd.DataFrame(krows)
kr.to_csv(root/'03_KNN/EXP-FT-KNN-002_handmade.csv',index=False)
pd.DataFrame({'true':y,**kpreds}).to_csv(root/'03_KNN/EXP-FT-KNN-002_predictions.csv',index=False)
print(kr[['condition','correct','macro_precision','macro_f1','corrected','regressed']].to_string(index=False))

### 원본 셀 13

## 4. KNN duplicate ablation

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell14
import sklearn.neighbors as nb
from final_knn_normal import sample_balanced,select_extra,BASE_FRACTION,SEED,WEAK_LABELS,ADDITIONAL_COUNT
rawh=np.concatenate([wr[w]['X'] for w in fb.W]); yh=np.concatenate([wr[w]['y'] for w in fb.W])
ix=fb.balanced_nested_indices(np.repeat(fb.W,640),yh,seed=fb.HAND_SEED)
qidx=ix[25]; ridx=np.setdiff1d(ix[50],ix[25],assume_unique=True)
sel=sample_balanced(off['y_train'],fraction=BASE_FRACTION,seed=SEED)
extra=select_extra(off['y_train'],sel,ADDITIONAL_COUNT,[labels.index(s) for s in WEAK_LABELS],seed=SEED+142)
# source group ID from actual official split + augmented lineages; existing official archive includes original source groups
groups=np.asarray(off['train_group_ids'],dtype=np.int64)
# use unique source groups, avoid collision with hand IDs (negative)
hands=-1-np.arange(2560,dtype=np.int64)
lineage=np.concatenate([groups,groups[sel],groups[extra],hands,hands,hands[qidx],hands[ridx]])
assert len(lineage)==len(ty)
assert np.array_equal(ty[107520:110080],yh) and np.array_equal(ty[110080:112640],yh)
assert np.array_equal(tr[112640:113280],tr[107520+qidx]) and np.array_equal(tr[113280:113920],tr[110080+ridx])
print('repeat rows and lineage check passed',len(np.unique(lineage)))
np.save(root/'03_KNN/train_lineage_ids.npy',lineage)
# repeat ablations
rrows=[]
rpreds={}
for name,ixr in [('R00',np.arange(len(ty))),('R01_remove_all',np.arange(112640)),('R02_original_only',np.arange(113280)),('R03_augmented_only',np.r_[np.arange(112640),np.arange(113280,113920)])]:
 t=time.perf_counter()
 if name=='R00': pred=kbase
 else:
  mm=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='euclidean',algorithm='brute',n_jobs=2).fit(tr[ixr],ty[ixr])
  pred=mm.predict(Xv);del mm
 rpreds[name]=pred
 rrows.append({'condition':name,'train_rows':len(ixr),'batch_eval_s':time.perf_counter()-t,**stats(pred,kbase)})
print(pd.DataFrame(rrows)[['condition','train_rows','correct','macro_f1','corrected','regressed']])

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell15
# R04: select distinct original source-group neighbors; adaptive expansion if needed.
t=time.perf_counter()
dl,il=knn.kneighbors(Xv,n_neighbors=32)
chosen_d=[];chosen_i=[]
for j,(drow,irow) in enumerate(zip(dl,il)):
 seen=set();pairs=[]
 for d,i in zip(drow,irow):
  g0=int(lineage[i])
  if g0 not in seen: pairs.append((d,i));seen.add(g0)
  if len(pairs)==3:break
 if len(pairs)<3:
  dd,ii=knn.kneighbors(Xv[j:j+1],n_neighbors=128)
  seen=set();pairs=[]
  for d,i in zip(dd[0],ii[0]):
   g0=int(lineage[i])
   if g0 not in seen:pairs.append((d,i));seen.add(g0)
   if len(pairs)==3:break
 assert len(pairs)==3
 chosen_d.append([z[0] for z in pairs]);chosen_i.append([z[1] for z in pairs])
pred=vote(np.asarray(chosen_d),np.asarray(chosen_i),3,1)
rpreds['R04_lineage_cap']=pred
rrows.append({'condition':'R04_lineage_cap','train_rows':len(ty),'batch_eval_s':time.perf_counter()-t,**stats(pred,kbase)})
pd.DataFrame(rrows).to_csv(root/'03_KNN/EXP-FT-KNN-003_handmade.csv',index=False)
pd.DataFrame({'true':y,**rpreds}).to_csv(root/'03_KNN/EXP-FT-KNN-003_predictions.csv',index=False)
print(pd.DataFrame(rrows)[['condition','correct','macro_precision','macro_f1','corrected','regressed']])

### 원본 셀 16

## 5. SVC full training feature reconstruction

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell17
t=time.perf_counter()
Xs,ys,meta=fb.train_features('SVC_Combined',off,wr)
print(Xs.shape,meta,time.perf_counter()-t)
np.save(root/'runtime/svc_X.npy',Xs)
np.save(root/'runtime/svc_y.npy',ys)
np.save(root/'runtime/dev_X.npy',Xv)
np.save(root/'runtime/dev_y.npy',y)
np.save(root/'runtime/official_X.npy',Xo)
np.save(root/'runtime/official_y.npy',yo)
print('memory',Path('/sys/fs/cgroup/memory.current').read_text())

### 원본 셀 18

## 6. SVC pilot setup

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell19
# Predeclare pilot for resource-limited SVC screen (explicit not full DOE). 
# Save exact training selections and augmentation candidate (no validation used in generation).
rng=np.random.default_rng(42)
# sample balanced 500 per class official, 4 per class/cohort per origin/aug, 2 per class/cohort for targeted
pilot=[]
for c0 in range(16):
 pilot.extend(rng.choice(np.flatnonzero(ys[:76800]==c0),500,replace=False))
for offset in [76800,79360]:
 for w0 in range(4):
  for c0 in range(16):
   pool=np.flatnonzero(yh[w0*640:(w0+1)*640]==c0)+w0*640
   pilot.extend(offset+rng.choice(pool,4,replace=False))
# exact target source indices used by baseline
rngt=np.random.default_rng(fb.TARGET_SEED)
target_groups=[]
for w in fb.W:
 for c0 in fb.TARGET_CLASSES:
  pool=np.flatnonzero((np.repeat(fb.W,640)==w)&(yh==c0))
  target_groups.append(rngt.permutation(pool)[:20])
ti=np.sort(np.concatenate(target_groups))
assert np.array_equal(ys[81920:],yh[ti])
# four-cohort target sampled 2 per class/cohort
for w in fb.W:
 for c0 in fb.TARGET_CLASSES:
  pool=np.flatnonzero((np.repeat(fb.W,640)[ti]==w)&(yh[ti]==c0))
  pilot.extend(81920+rng.choice(pool,2,replace=False))
pilot=np.sort(np.array(pilot))
assert len(pilot)==8528
# predeclared pilot official eval balanced 40/class =640 (not full)
oid=np.sort(np.concatenate([rng.choice(np.flatnonzero(yo==c0),40,replace=False) for c0 in range(16)]))
np.savez(root/'02_SVC/pilot_indices.npz',training_rows=pilot,official_validation_rows=oid)
proto['SVC']['pilot_resource_adaptation']={'rows':len(pilot),'official_train':8000,'handmade_original':256,'handmade_augmented':256,'handmade_targeted':16,'official_validation':640,'handmade_validation':1280,'purpose':'screening only; full 9x DOE not claimed complete; full fit required for adoption'}
(root/'EXPERIMENT_PROTOCOL.json').write_text(json.dumps(proto,ensure_ascii=False,indent=2))
print(len(pilot),len(oid),np.bincount(ys[pilot]))
# set up data 
Xp=Xs[pilot];yp=ys[pilot];hp=(pilot>=76800)
gscale=1/(Xp.shape[1]*Xp.var())
print('pilot gamma',gscale)

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell20
from sklearn.svm import SVC
spdir=root/'02_SVC'
sprows=[]
sppreds={}
def svc_pilot(name,exp,C=1.,gm=1.,hand_w=1.,target_w=1.,xx=Xp,yy=yp,handmask=hp):
    t=time.perf_counter()
    gamma=float(gm/(xx.shape[1]*xx.var()))
    weights=np.ones(len(yy),np.float64); weights[handmask]*=hand_w
    weights[handmask & np.isin(yy,[labels.index(c) for c in 'BDC'])]*=target_w
    m=SVC(C=C,gamma=gamma,cache_size=200)
    m.fit(xx,yy,sample_weight=weights)
    fit=time.perf_counter()-t
    t=time.perf_counter(); pv=m.predict(Xv); vo=m.predict(Xo[oid]); infer=time.perf_counter()-t
    res={'condition':name,'experiment':exp,'scope':'pilot','train_rows':len(yy),'C':C,'gamma_multiplier':gm,'gamma':gamma,'hand_weight':hand_w,'target_weight':target_w,'fit_s':fit,'batch_eval_s':infer,'official_pilot_n':len(oid),'official_pilot_correct':int((vo==yo[oid]).sum()),**stats(pv,sbase)}
    sprows.append(res);sppreds[name]=pv
    pd.DataFrame(sprows).to_csv(spdir/'pilot_all_conditions.csv',index=False)
    pd.DataFrame({'true':y,**sppreds}).to_csv(spdir/'pilot_predictions.csv',index=False)
    print(name,res['correct'],round(res['macro_f1'],5),res['official_pilot_correct'],round(fit,2),flush=True)
    return res
for c in [1,.5,2]:
 svc_pilot(f'C{c}_G1','001',C=c)

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell21
for c in [.5,1,2]:
 for g0 in [.5,2]:
  svc_pilot(f'C{c}_G{g0}','001',C=c,gm=g0)

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell22
for w in [1.5,2,3,4]:
 svc_pilot(f'SW{w}','002',hand_w=w)
for w in [1.5,2]:
 svc_pilot(f'TW{w}','003',target_w=w)

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell23
# Additional augmentation controlled comparison: 80 selected rows instead of 160, or +240 new B/D/C variants.
# Baseline indices, reduction pilot exactly one retained from two per B/D cohort, augmentation generation from existing training only.
rng2=np.random.default_rng(20261128)
target_base_pilot=pilot[pilot>=81920]
keep_red=[]
for w in fb.W:
 for c0 in fb.TARGET_CLASSES:
  cand=target_base_pilot[(np.repeat(fb.W,640)[ti[target_base_pilot-81920]]==w)&(ys[target_base_pilot]==c0)]
  assert len(cand)==2
  keep_red.append(cand[0])
rp=np.r_[pilot[pilot<81920],np.sort(keep_red)]
svc_pilot("TA_reduced","003",xx=Xs[rp],yy=ys[rp],handmask=rp>=76800)
# prepare full new target augment 20 per cohort per B,D,C, distinct random variant seed; preserve existing base
add_groups=[]
for w in fb.W:
 for c0 in [labels.index(c) for c in 'BDC']:
  pool=np.flatnonzero((np.repeat(fb.W,640)==w)&(yh==c0))
  add_groups.append(rng2.permutation(pool)[:20])
add_idx=np.sort(np.concatenate(add_groups))
add_feat=fb.hand_aug(rawh,add_idx,20261128+1000)
add_y=yh[add_idx]
assert add_feat.shape==(240,1108)
np.savez(root/'runtime/svc_target_extra.npz',X=add_feat,y=add_y,source_indices=add_idx)
addpilot=[]
for w in fb.W:
 for c0 in [labels.index(c) for c in 'BDC']:
  pool=np.flatnonzero((np.repeat(fb.W,640)[add_idx]==w)&(add_y==c0))
  addpilot.extend(rng2.choice(pool,2,replace=False))
addpilot=np.sort(addpilot)
svc_pilot("TA_increased","003",xx=np.vstack([Xp,add_feat[addpilot]]),yy=np.r_[yp,add_y[addpilot]],handmask=np.r_[hp,np.ones(len(addpilot),bool)])
# Record actual same-scope pilot stats vs pilot baseline
sp=pd.DataFrame(sprows); basep=sp[sp.condition=='C1_G1'].iloc[0]
sp['delta_correct_vs_pilot']=sp.correct-basep.correct
sp['delta_f1_vs_pilot']=sp.macro_f1-basep.macro_f1
sp['pilot_guardrail']= (sp.correct>basep.correct)&(sp.macro_precision>=basep.macro_precision)&(sp.macro_f1>basep.macro_f1)&(sp.official_pilot_correct>=basep.official_pilot_correct-1)
sp.to_csv(spdir/'pilot_all_conditions.csv',index=False)
print(sp[['condition','correct','macro_precision','macro_f1','official_pilot_correct','pilot_guardrail']].to_string(index=False))

### 원본 셀 24

## 7. KNN distance-function pilot

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell25
# KNN metric pilot - balanced 500 per class training, 20 per class for each dev set.
# Pilot labels are existing train only; no new validation samples are used in fit.
rngk=np.random.default_rng(42)
kp=np.sort(np.concatenate([rngk.choice(np.flatnonzero(ty==c0),500,replace=False) for c0 in range(16)]))
kd=np.sort(np.concatenate([rngk.choice(np.flatnonzero(y==c0),20,replace=False) for c0 in range(16)]))
ko=np.sort(np.concatenate([rngk.choice(np.flatnonzero(yo==c0),20,replace=False) for c0 in range(16)]))
np.savez(root/'03_KNN/EXP-FT-KNN-001_pilot_indices.npz',training_rows=kp,handmade_validation_rows=kd,official_validation_rows=ko)
xx=tr[kp]; yy=ty[kp]
xxv=np.vstack([Xv[kd],Xo[ko]])
yyv=np.r_[y[kd],yo[ko]]
metric_rows=[]
metric_predictions={}
def p_pilot(pv):
 t=time.perf_counter()
 m=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='minkowski',p=pv,algorithm='brute',n_jobs=2).fit(xx,yy)
 pt=m.predict(xxv)
 dur=time.perf_counter()-t
 metric_predictions[str(pv)]=pt
 ss=stats(pt[:len(kd)],kbase[kd],y=y[kd])
 rec={'p':pv,'scope':'pilot','train_rows':len(kp),'official_n':len(ko),'official_correct':int((pt[len(kd):]==yo[ko]).sum()),'elapsed_batch_s':dur,**ss}
 metric_rows.append(rec)
 pd.DataFrame(metric_rows).to_csv(root/'03_KNN/EXP-FT-KNN-001_pilot.csv',index=False)
 print(pv,rec['correct'],rec['macro_f1'],rec['official_correct'],dur,flush=True)
for pv in [2,1]:
 p_pilot(pv)

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell26
def p_pilot_chunk(pv,lo,hi):
 m=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='minkowski',p=pv,algorithm='brute',n_jobs=2).fit(xx,yy)
 t=time.perf_counter();pred=m.predict(xxv[lo:hi]);sec=time.perf_counter()-t
 print(pv,lo,hi,'seconds',sec,flush=True)
 return pred,sec
d15_1,t15_1=p_pilot_chunk(1.5,0,320)

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell27
d15_2,t15_2=p_pilot_chunk(1.5,320,640)
pt=np.r_[d15_1,d15_2]
metric_predictions['1.5']=pt
metric_rows.append({'p':1.5,'scope':'pilot','train_rows':len(kp),'official_n':len(ko),'official_correct':int((pt[len(kd):]==yo[ko]).sum()),'elapsed_batch_s':t15_1+t15_2,**stats(pt[:len(kd)],kbase[kd],y=y[kd])})
pd.DataFrame(metric_rows).to_csv(root/'03_KNN/EXP-FT-KNN-001_pilot.csv',index=False)
print(metric_rows[-1]['correct'],metric_rows[-1]['macro_f1'],metric_rows[-1]['official_correct'])

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell28
d3_1,t3_1=p_pilot_chunk(3,0,320)

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell29
d3_2,t3_2=p_pilot_chunk(3,320,640)
pt=np.r_[d3_1,d3_2];metric_predictions['3']=pt
metric_rows.append({'p':3,'scope':'pilot','train_rows':len(kp),'official_n':len(ko),'official_correct':int((pt[len(kd):]==yo[ko]).sum()),'elapsed_batch_s':t3_1+t3_2,**stats(pt[:len(kd)],kbase[kd],y=y[kd])})
mp=pd.DataFrame(metric_rows);mp['promote']=(mp.correct>mp.correct.iloc[0])&(mp.macro_f1>mp.macro_f1.iloc[0])
mp.to_csv(root/'03_KNN/EXP-FT-KNN-001_pilot.csv',index=False)
pd.DataFrame({'true':yyv,**metric_predictions}).to_csv(root/'03_KNN/EXP-FT-KNN-001_pilot_predictions.csv',index=False)
print(mp[['p','correct','macro_f1','official_correct','elapsed_batch_s','promote']])

### 원본 셀 30

## 8. KNN PCA full training

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell31
from sklearn.decomposition import PCA
t=time.perf_counter()
pc=PCA(n_components=512,svd_solver='covariance_eigh',whiten=False)
pc.fit(tr)
pca_fit_sec=time.perf_counter()-t
print('PCA fit',pca_fit_sec,pc.explained_variance_ratio_.sum())
joblib.dump(pc,root/'03_KNN/pca512_training_only.joblib',compress=3)
np.savez(root/'03_KNN/pca_variance.npz',ratios=pc.explained_variance_ratio_)
print('memory',Path('/sys/fs/cgroup/memory.current').read_text())

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell32
# PCA full training projection; nested prefixes share exact fitted eigenspace.
t=time.perf_counter()
zt=pc.transform(tr);zv=pc.transform(Xv);zo=pc.transform(Xo)
pca_transform_sec=time.perf_counter()-t
print(zt.shape,zt.dtype,pca_transform_sec)
print('memory',Path('/sys/fs/cgroup/memory.current').read_text())

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell33
pca_rows=[{'condition':'PCA00','dims':1108,'explained_var':1.0,'scope':'full_training_full_handmade',**stats(kbase,kbase)}]
pca_preds={'PCA00':kbase}
for dd in [512,256,128,64]:
    ztrain=np.ascontiguousarray(zt[:,:dd]);zzv=np.ascontiguousarray(zv[:,:dd])
    t=time.perf_counter()
    mod=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='euclidean',algorithm='brute',n_jobs=2).fit(ztrain,ty)
    ft=time.perf_counter()-t;t=time.perf_counter();pred=mod.predict(zzv);et=time.perf_counter()-t
    rec={'condition':f'PCA{dd}','dims':dd,'explained_var':float(pc.explained_variance_ratio_[:dd].sum()),'scope':'full_training_full_handmade','pca_fit_s_shared':pca_fit_sec,'pca_transform_512_s_shared':pca_transform_sec,'knn_fit_s':ft,'batch_eval_s':et,**stats(pred,kbase)}
    pca_rows.append(rec);pca_preds[f'PCA{dd}']=pred
    print(rec['condition'],rec['correct'],rec['macro_f1'],rec['explained_var'],et,flush=True)
    del ztrain,zzv,mod
    gc.collect()
pd.DataFrame(pca_rows).to_csv(root/'03_KNN/EXP-FT-KNN-004_handmade.csv',index=False)
pd.DataFrame({'true':y,**pca_preds}).to_csv(root/'03_KNN/EXP-FT-KNN-004_predictions.csv',index=False)

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell34
# Confirm official development score for PCA alternatives with equal accuracy;
# guardrail is separate from strict accuracy gate.
for dd in [128,512]:
 ztrain=np.ascontiguousarray(zt[:,:dd])
 mm=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='euclidean',algorithm='brute',n_jobs=2).fit(ztrain,ty)
 t=time.perf_counter()
 pp=[]
 for lo in range(0,len(yo),2400):pp.append(mm.predict(np.ascontiguousarray(zo[lo:lo+2400,:dd])))
 po=np.concatenate(pp);duration=time.perf_counter()-t
 row=next(r for r in pca_rows if r['dims']==dd)
 row['official_correct']=int((po==yo).sum());row['official_accuracy']=float(np.mean(po==yo));row['official_guardrail_pass']=row['official_correct']>=18856-19
 np.save(root/f'03_KNN/PCA{dd}_official_pred.npy',po)
 print(dd,row['official_correct'],row['official_guardrail_pass'],duration,flush=True)
 del ztrain,mm
 gc.collect()
pd.DataFrame(pca_rows).to_csv(root/'03_KNN/EXP-FT-KNN-004_handmade.csv',index=False)

### 원본 셀 35

## 9. KNN timing screen

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell36
# Timing screen on a fixed 128-image subset, three passes, full preprocessing, same affinity and environment for all.
# Not a replacement for final 1280x5 timing.
rngtime=np.random.default_rng(42)
tim_idx=np.sort(np.concatenate([rngtime.choice(np.flatnonzero(y==c0),8,replace=False) for c0 in range(16)]))
np.save(root/'03_KNN/latency_screen_indices.npy',tim_idx)
tim_rows=[]
# baseline
def latency_screen(m,dd):
 def one(i):
  f=fb.hybrid(raw[i:i+1])
  if dd!=1108:f=pc.transform(f)[:,:dd].copy()
  return int(m.predict(f)[0])
 for i in tim_idx[:16]:one(i)
 reps=[]
 ref=None
 for rep in range(3):
  t=time.perf_counter();pred=np.array([one(i) for i in tim_idx]);elapsed=time.perf_counter()-t
  reps.append(elapsed*1000/len(tim_idx))
  if ref is None:ref=pred
  else:assert np.array_equal(ref,pred)
 return reps,ref
for dd in [1108,128,512]:
 if dd==1108:mm=knn
 else:mm=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='euclidean',algorithm='brute',n_jobs=2).fit(np.ascontiguousarray(zt[:,:dd]),ty)
 times,pred=latency_screen(mm,dd)
 ref=pca_preds['PCA00' if dd==1108 else f'PCA{dd}'][tim_idx]
 rec={'dims':dd,'n':len(tim_idx),'repeats':3,'mean_ms_per_image':float(np.mean(times)),'individual_ms':times,'matches_batch':int(np.sum(pred==ref)),'timing_scope':'screen only, same machine; record concurrent workload status'}
 tim_rows.append(rec)
 print(rec,flush=True)
 if dd!=1108:del mm
pd.DataFrame(tim_rows).to_csv(root/'03_KNN/PCA_latency_screen.csv',index=False)

### 원본 셀 37

## 10. KNN PCA vote recheck

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell38
# Recheck vote settings after PCA for the two equal-accuracy speed candidates.
pvrows=[];pvp={}
for dd in [128,512]:
 zztr=np.ascontiguousarray(zt[:,:dd]);zzv=np.ascontiguousarray(zv[:,:dd])
 mm=nb.KNeighborsClassifier(n_neighbors=3,weights='distance',metric='euclidean',algorithm='brute',n_jobs=2).fit(zztr,ty)
 dp,ip=mm.kneighbors(zzv,n_neighbors=9)
 assert np.array_equal(vote(dp,ip,3,1),pca_preds[f'PCA{dd}'])
 for k in [1,3,5,7,9]:
  for al in ([1] if k==1 else [.5,1,2]):
   pr=vote(dp,ip,k,al)
   tag=f'PCA{dd}_k{k}_a{al}'
   pvrows.append({'condition':tag,'dims':dd,'k':k,'alpha':al,**stats(pr,kbase)});pvp[tag]=pr
 del zztr,zzv,mm
 gc.collect()
pd.DataFrame(pvrows).to_csv(root/'03_KNN/EXP-FT-KNN-005_pca_vote_recheck.csv',index=False)
pd.DataFrame({'true':y,**pvp}).to_csv(root/'03_KNN/EXP-FT-KNN-005_predictions.csv',index=False)
print(pd.DataFrame(pvrows).sort_values(['correct','macro_f1'],ascending=False)[['condition','correct','macro_precision','macro_f1']].head(12).to_string(index=False))

### 원본 셀 39

## 11. Full SVC candidate training
Copy the supplied `run_svc_full.py`, `run_svc_remaining.py`, and `run_svc_sw15_followup.py` into the reproduction output directory (the scripts expect `runtime/*.npy` prepared above).
Run separately from the notebook after releasing large intermediate arrays; each fit may take substantial CPU time. The first script originally attempted 3 candidates and was stopped after SW4's checkpoint and dev predictions. Remaining scripts use an explicitly cross-checked RBF OVO formula for official evaluation to avoid expensive redundant computation.
Do not run candidates selected only by pilot as final models without the guardrails. The exact actual outputs are in the original result package. For prediction extraction from a candidate model, load the joblib file and apply the original `final_benchmark.hybrid` feature function.

In [ ]:
%%archive_source 07_fine_tuning/Fine_Tuning_Reproduction.ipynb#cell40
# Example inference extraction with an already fitted checkpoint (not a fit):
import joblib
# m=joblib.load('path/to/model.joblib')
# prediction=m.predict(fb.hybrid(raw.reshape(-1,784)))
# pd.DataFrame({'collector':v['collector_id'],'index':v['sample_index'],'true':y,'pred':prediction}).to_csv(root/'candidate_predictions.csv',index=False)


## 실험 실행 및 결과 확인

`RUN_PLAN`에 실제 코드의 CLI 인자와 실행 순서를 넣고, `ALLOW_EXECUTION=True`를 설정한 경우에만 실행된다. 이전 결과를 다시 계산한 것과 새 학습 결과를 구별한다.

In [ ]:
if RUN_PLAN:
    for source_path, arguments in RUN_PLAN:
        execute_original(source_path, arguments)
else:
    print("원본 코드 등록 완료. 학습/평가는 실행하지 않았습니다.")